# Rearranging Reports

## inspecting

In [7]:
import hashlib
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import pandas as pd

# ---------- Settings ----------
ROOT = Path.cwd()
if not (ROOT / "reports").exists():      # in case the notebook runs from notebooks/
    ROOT = ROOT.parent

SCAN_DIRS = [
    ROOT / "reports" / "transformer",
    ROOT / "reports" / "transformer_stage1",
    ROOT / "reports" / "transformer_stage2",
]

def sha256_of(path: Path, chunk=1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

# ---------- 1) Collect all files ----------
rows = []
for base in SCAN_DIRS:
    if not base.exists():
        print(f"WARNING: folder does not exist: {base}")
        continue
    for p in sorted(base.rglob("*")):
        if p.is_file():
            st = p.stat()
            rows.append({
                "root": base.name,
                "subfolder": str(p.parent.relative_to(base)) or ".",
                "file": p.name,
                "size_bytes": st.st_size,
                "size_MB": round(st.st_size / 1e6, 3),
                "modified": datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M"),
                "path": p,
            })

df = pd.DataFrame(rows)
if df.empty:
    raise SystemExit("No files found; check the ROOT path.")

# ---------- 2) Hash only files that share a size with another file ----------
# Different size => different content, so those files cannot be duplicates.
# Names are never used for matching, only content.
by_size = defaultdict(list)
for i, s in df["size_bytes"].items():
    by_size[s].append(i)

df["sha256"] = None
for s, idxs in by_size.items():
    if len(idxs) > 1:
        for i in idxs:
            df.at[i, "sha256"] = sha256_of(df.at[i, "path"])

# ---------- 3) Group duplicates ----------
hash_counts = df["sha256"].value_counts()
dup_hashes = hash_counts[hash_counts > 1].index.tolist()
group_map = {h: f"D{n:02d}" for n, h in enumerate(dup_hashes, 1)}
df["dup_group"] = df["sha256"].map(group_map)
df["sha_short"] = df["sha256"].str[:10]

# ---------- 4) Print tables ----------
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 60)
cols = ["root", "subfolder", "file", "size_MB", "modified", "dup_group", "sha_short"]

print("=" * 100)
print("1) Summary per root folder")
print("=" * 100)
summary = df.groupby("root").agg(files=("file", "count"), total_MB=("size_MB", "sum")).round(2)
print(summary.to_string())

print("\n" + "=" * 100)
print("2) Files that exist ONLY in transformer_stage1 (unique content, not found elsewhere)")
print("=" * 100)
only_s1 = df[(df["root"] == "transformer_stage1") & (df["dup_group"].isna())]
print(only_s1.sort_values(["subfolder", "file"])[cols].fillna("").to_string(index=False)
      if not only_s1.empty else "None: every stage1 file has a duplicate elsewhere.")

print("\n" + "=" * 100)
print("3) Duplicate groups (byte-identical content, names may differ)")
print("=" * 100)
dups = df[df["dup_group"].notna()].sort_values(["dup_group", "root", "subfolder", "file"])
if dups.empty:
    print("No duplicate files found.")
else:
    print(dups[["dup_group"] + cols[:5] + ["sha_short"]].to_string(index=False))
    wasted = sum(g["size_bytes"].iloc[0] * (len(g) - 1) for _, g in dups.groupby("dup_group"))
    print(f"\n{dups['dup_group'].nunique()} duplicate groups | "
          f"{len(dups)} files involved | wasted space ~ {wasted/1e6:.2f} MB")

print("\n" + "=" * 100)
print("4) Duplicate groups where the file NAMES differ (renamed copies)")
print("=" * 100)
renamed = dups.groupby("dup_group").filter(lambda g: g["file"].nunique() > 1)
print(renamed[["dup_group"] + cols[:5]].to_string(index=False)
      if not renamed.empty else "None.")

print("\n" + "=" * 100)
print("5) Same file name with DIFFERENT content (overwrite/collision risk)")
print("=" * 100)
same_name = df.groupby("file").filter(lambda g: len(g) > 1)
same_name = same_name.groupby("file").filter(
    lambda g: g["sha256"].isna().any() or g["sha256"].nunique() > 1
)
print(same_name.sort_values(["file", "root"])[cols].fillna("").to_string(index=False)
      if not same_name.empty else "None found.")

print("\n" + "=" * 100)
print("6) Which roots share each duplicate group (overlap matrix)")
print("=" * 100)
if not dups.empty:
    overlap = (dups.groupby("dup_group")["root"]
               .apply(lambda s: " + ".join(sorted(s.unique()))))
    print(overlap.value_counts().rename("groups").to_frame().to_string())

1) Summary per root folder
                    files  total_MB
root                               
transformer            24     18.44
transformer_stage1      8     42.29
transformer_stage2      3      5.83

2) Files that exist ONLY in transformer_stage1 (unique content, not found elsewhere)
              root    subfolder                              file  size_MB         modified dup_group sha_short
transformer_stage1 full/figures     01_stage1_learning_curves.png    6.536 2026-11-19 04:28                    
transformer_stage1 full/figures            02_stage1_vs_tfidf.png    6.264 2026-11-19 04:23                    
transformer_stage1 full/figures       03_stage1_confusion_raw.png    8.631 2026-11-19 04:43                    
transformer_stage1 full/figures 04_stage1_confusion_corrected.png    8.630 2026-11-19 04:43                    
transformer_stage1 full/figures        05_stage1_per_class_f1.png    6.397 2026-11-19 04:43                    
transformer_stage1 full/figures    

/tmp/ipykernel_10195/1133041353.py:84: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  print(only_s1.sort_values(["subfolder", "file"])[cols].fillna("").to_string(index=False)
